In [7]:
import cv2
import time  # <--- NEW: Imported time for timing calculations
from ultralytics import YOLO
import numpy as np

# --- Configuration ---
MODEL_PATH = 'fire.pt'
CONFIDENCE_THRESHOLD = 0.50 # Only show detections the model is at least 50% sure about

# --- Load YOLO Model ---
try:
    model = YOLO(MODEL_PATH)
    print(f"✅ Successfully loaded YOLO model: {MODEL_PATH}")
except Exception as e:
    print(f"❌ FATAL ERROR: Could not load YOLO model. Ensure '{MODEL_PATH}' is in the same directory.")
    print(f"Error details: {e}")
    exit()

# --- CRITICAL FIX: Robust Camera Initialization ---
cap = None
camera_opened = False
print("Attempting to open webcam (trying indices 0, 1, 2, 3)...")

for index in range(0, 4): 
    # Use a high-quality backend (CAP_DSHOW for Windows) if available
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) 
    
    if cap.isOpened():
        print(f"✅ Successfully opened webcam using index {index}")
        camera_opened = True
        break
    
    if cap:
        cap.release() 

if not camera_opened:
    print("❌ FATAL ERROR: Could not open any webcam. Please check device connection and privacy settings.")
    exit()
# --- END CRITICAL FIX ---


# --- Main Detection Loop ---
print("Starting real-time detection. Press 'q' in the video window to stop.")

# --- NEW: Timing Initialization ---
prev_frame_time = time.time()
# --- END NEW ---

while True:
    
    # --- NEW: Start time measurement ---
    t1 = time.time()
    # --- END NEW ---
    
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame.")
        break
    
    # 1. Run inference on the frame
    results = model.predict(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
    
    # 2. Get the annotated frame
    annotated_frame = results[0].plot()

    # --- NEW: FPS and Latency Calculation ---
    current_frame_time = time.time()
    
    # Latency: Time taken for one full loop cycle (capture, prediction, display)
    latency_sec = current_frame_time - prev_frame_time
    
    # FPS: Frames per second (1 / latency)
    fps = 1.0 / latency_sec if latency_sec > 0 else 0
    
    # Prepare text strings
    fps_text = f'FPS: {fps:.1f}'
    latency_text = f'Latency: {latency_sec*1000:.1f}ms' # Convert to milliseconds
    
    # Update previous time for the next loop
    prev_frame_time = current_frame_time
    # --- END NEW ---

    # --- NEW: Drawing FPS and Latency ---
    # Draw FPS in the top left corner
    cv2.putText(annotated_frame, fps_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)
    
    # Draw Latency just below FPS
    cv2.putText(annotated_frame, latency_text, (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2, cv2.LINE_AA)
    # --- END NEW ---

    # 3. Display the frame
    cv2.imshow("Fire Detection", annotated_frame)

    # 4. Exit the program if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# --- Cleanup ---
cap.release()
cv2.destroyAllWindows()

✅ Successfully loaded YOLO model: fire.pt
Attempting to open webcam (trying indices 0, 1, 2, 3)...
✅ Successfully opened webcam using index 0
Starting real-time detection. Press 'q' in the video window to stop.
